# 04 — Chi-Square Bad-Data Detection

本节引入 Chi-square 阈值，将正常数据、普通攻击和结构化攻击放在同一个检测框架下比较。

In [1]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [2]:
from src.grid_model import build_three_bus_model, generate_measurement
from src.state_estimator import wls_estimate, chi_square_threshold, bad_data_alarm
from src.attack_generator import naive_attack, structured_fdi_attack

rng = np.random.default_rng(42)
model = build_three_bus_model(sigma=0.01)

m, n = model.H.shape
threshold = chi_square_threshold(m, n, false_alarm_rate=0.05)
print("Chi-square threshold =", threshold)


Chi-square threshold = 7.814727903251179


In [3]:
z, _ = generate_measurement(model, rng)

normal = wls_estimate(z, model.H, model.R)

z_naive, _ = naive_attack(z, measurement_index=0, magnitude=0.10)
naive = wls_estimate(z_naive, model.H, model.R)

c = np.array([0.005, -0.004])
z_structured, _ = structured_fdi_attack(z, model.H, c)
structured = wls_estimate(z_structured, model.H, model.R)

cases = {
    "Normal": normal.J,
    "Naive FDI": naive.J,
    "Structured FDI": structured.J,
}

for name, J in cases.items():
    print(f"{name:15s} J={J:10.6f}  alarm={bad_data_alarm(J, threshold)}")


Normal          J=  2.871034  alarm=False
Naive FDI       J= 71.198096  alarm=True
Structured FDI  J=  2.871034  alarm=False


In [4]:
names = list(cases.keys())
values = list(cases.values())

plt.figure(figsize=(7, 4))
plt.bar(names, values)
plt.axhline(threshold, linestyle="--", label="Chi-square threshold")
plt.ylabel("Residual statistic J")
plt.title("Residual-based bad-data detection")
plt.legend()
plt.tight_layout()
plt.show()


/tmp/ipykernel_913/1103377250.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Discussion

这个实验要让学生区分两个概念：

- **攻击幅度大**；
- **攻击是否与量测模型结构一致**。

传统 residual-only BDD 对前者敏感，但在理想线性条件下并不能保证识别 \(a=Hc\) 这类结构化扰动。
